# Gender wage gap in the US, 1976-2025
Data: CPS ASEC from IPUMS. Weight: `asecwt`.

This first part connects Google Drive, loads the data, and prints the shape, the column names, and a summary table.

## 1. Libraries

In [ ]:
# Colab already has pandas, numpy, and matplotlib, so no pip install is needed
import pandas as pd  # tables
import numpy as np  # math helpers for later steps
import matplotlib.pyplot as plt  # charts for later steps
from google.colab import drive  # connect Google Drive

## 2. Settings
Change the path or the column list here.

In [ ]:
# Path to the raw data file (change here if the file moves)
data_path = "/content/drive/My Drive/jobs_assignment_data_20261104.csv"  # file in the root of My Drive

# Columns we need from the 1.6 GB file (the file has 85 columns)
needed_columns = [  # start of the list
    "year", "asecwt",  # income year and person-level survey weight
    "female", "fulltime",  # sex and full-time flag
    "age", "agegrp",  # age and age group
    "wkswork", "hrswk",  # weeks worked last year and usual hours per week
    "incwage_real", "hrwage_real",  # yearly wage and hourly wage in 2025 dollars
    "educ5", "raceeth", "marst3",  # education, race and ethnicity, marital status
    "nchild_cat", "child_u5",  # number of children, child under 5
    "occ6", "classwkr3", "metro",  # occupation, class of worker, metro area
]  # end of the list

# Show decimal numbers with 1 decimal and a comma for thousands
pd.options.display.float_format = "{:,.1f}".format  # display format for floats

## 3. Connect Google Drive

In [ ]:
drive.mount("/content/drive")  # connect Drive (click Allow when it asks)

## 4. Column names in the file

In [ ]:
file_columns = pd.read_csv(data_path, nrows=0).columns.tolist()  # read only the header row
print("Number of columns in the file:", len(file_columns))  # how many columns the file has
print("Column names:")  # label for the list
print(file_columns)  # every column name in the file
missing_names = [name for name in needed_columns if name not in file_columns]  # names we want but the file lacks
print("Needed columns not in the file:", missing_names)  # should be an empty list

## 5. Load the data and print the shape
We read only the columns we need. The raw table stays as loaded. We work on a copy.

In [ ]:
raw_data = pd.read_csv(data_path, usecols=needed_columns)  # read only the columns we need
data = raw_data.copy()  # work on a copy, the raw table stays as loaded
print("Shape of the file (rows, columns):", (len(data), len(file_columns)))  # every row is loaded, so rows match the file
print("Shape of the table we loaded (rows, columns):", data.shape)  # rows and the columns we kept

## 6. Summary table
One row for every column we loaded. The mean is weighted with `asecwt`. The count (n), the missing count, the min, and the max are unweighted. Numbers show 1 decimal.

For a 0/1 column like `female`, the mean is the share. For a code with several groups like `educ5`, the mean has no meaning by itself. We keep the row for n, missing, min, and max.

In [ ]:
summary_columns = needed_columns  # one row for every column we loaded
summary_rows = []  # list that will hold one row per column
for column in summary_columns:  # go through each column
    has_value = data[column].notna()  # True where the value is not missing
    values = data.loc[has_value, column]  # the values that are not missing
    weights = data.loc[has_value, "asecwt"]  # the weights for those people
    weighted_mean = (values * weights).sum() / weights.sum()  # mean weighted with asecwt
    summary_rows.append({  # add one row to the list
        "column": column,  # name of the column
        "n (unweighted)": has_value.sum(),  # people with a value
        "missing": (~has_value).sum(),  # people without a value
        "weighted mean": weighted_mean,  # average weighted with asecwt
        "min": values.min(),  # smallest value
        "max": values.max(),  # largest value
    })  # end of this row
summary_table = pd.DataFrame(summary_rows).set_index("column")  # turn the list into a table
print("Summary table (mean weighted with asecwt)")  # title above the table
print(summary_table.round(1))  # show the table with 1 decimal